# ATS-169 / ATS-175: Study 0 — Unified Benchmark (Colab GPU)

Run the full Study 0 benchmark on a Colab GPU in a single notebook:

1. **Baselines** — XGBoost, Logistic Regression (full + traditional), GBT, FT-Transformer (default)
2. **FT-Transformer hyperparameter sweep** — Grid search over lr × n_layers × d_token (48 configs)
3. **SSL Pretraining with tuned architecture** — Re-pretrain with best hyperparams at mask ratios 0.15, 0.30, 0.50
4. **Final comparison & Go/No-Go Gate** — All models head-to-head with bootstrap CIs

**Gate criterion:** FT-Transformer beats XGBoost on ≥3/5 distress outcomes by ≥0.01 AUROC.

**Outcomes:** stock_decline, earnings_restate, audit_qualification, sec_enforcement, bankruptcy

**Splits:** train ≤2017 | val 2018–2019 | test 2020–2023

**Prerequisites:**
1. Runtime → Change runtime type → **T4 GPU** (or A100 if available)
2. Data uploaded to Google Drive at `My Drive/Fin-JEPA/data/` with subdirs `raw/` and `processed/`
3. GitHub PAT stored in Colab Secrets as `GITHUB_PAT`

## 1. Environment Setup

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from google.colab import userdata
import os

# Retrieve the GitHub token from Colab Secrets
github_token = userdata.get('GITHUB_PAT')

username = "elroy-galbraith"
repository = "fin-jepa"
repo_url = f"https://{github_token}@github.com/{username}/{repository}.git"

if not os.path.exists(f'/content/{repository}'):
    !git clone {repo_url} /content/{repository}

%cd /content/{repository}
!pip install -q -e '.[dev]'

In [ ]:
# Symlink data from Google Drive so relative paths work
import os

DRIVE_DATA = '/content/drive/MyDrive/Fin-JEPA/data'

for subdir in ['raw', 'processed']:
    target = f'data/{subdir}'
    if os.path.islink(target) or os.path.isdir(target):
        !rm -rf {target}
    os.symlink(f'{DRIVE_DATA}/{subdir}', target)

# Verify data exists
!ls -la data/raw/xbrl_features.parquet
!ls -la data/processed/label_database.parquet
!ls -la data/raw/market/market_aligned.parquet
print('Data linked successfully!')

In [ ]:
# Verify GPU
import torch
assert torch.cuda.is_available(), 'No GPU detected! Change runtime to T4 or A100.'
print(f'GPU: {torch.cuda.get_device_name(0)}')
props = torch.cuda.get_device_properties(0)
print(f'VRAM: {props.total_mem / 1e9:.1f} GB')

In [ ]:
# Common setup: logging, configs, control flag
import json
import logging
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from omegaconf import OmegaConf

logging.basicConfig(level=logging.INFO, format='%(levelname)s %(name)s: %(message)s')

# Set to True to re-run experiments even if cached results exist
FORCE_RERUN = False

# Load configs from YAML (override individual values below if needed)
benchmark_cfg = OmegaConf.load('configs/study0/benchmark.yaml')
ssl_cfg = OmegaConf.load('configs/study0/ssl_experiment.yaml')

# Paths
RESULTS_DIR = Path('results/study0')
BENCHMARK_PATH = RESULTS_DIR / 'benchmark_results.json'
SWEEP_PATH = RESULTS_DIR / 'ft_sweep_results.json'
SSL_V2_PATH = RESULTS_DIR / 'ssl_experiment_results_v2.json'
FINAL_PATH = RESULTS_DIR / 'final_benchmark.json'

OUTCOMES = ['stock_decline', 'earnings_restate', 'audit_qualification',
            'sec_enforcement', 'bankruptcy']

print(f'Results dir: {RESULTS_DIR}')
print(f'Benchmark cached: {BENCHMARK_PATH.exists()}')
print(f'Sweep cached: {SWEEP_PATH.exists()}')
print(f'SSL v2 cached: {SSL_V2_PATH.exists()}')
print(f'FORCE_RERUN: {FORCE_RERUN}')

## 2. Baseline Benchmark

Train all baseline models on each distress outcome:
- **XGBoost** (full features)
- **Logistic Regression** — full features + traditional ratios only
- **GBT** (raw XBRL features)
- **FT-Transformer** from scratch (default hyperparams)

Also evaluates the built-in go/no-go gate (FT-Transformer vs XGBoost).

In [ ]:
%%time
if not FORCE_RERUN and BENCHMARK_PATH.exists():
    print(f'Loading cached baseline results from {BENCHMARK_PATH}')
    with open(BENCHMARK_PATH) as f:
        benchmark_results = json.load(f)
else:
    from fin_jepa.training.train_study0 import run_benchmark
    benchmark_results = run_benchmark(benchmark_cfg)
    print('Baseline benchmark complete.')

# Normalize format: run_benchmark returns {gate: ..., outcomes: ...}
if 'outcomes' in benchmark_results:
    benchmark_outcomes = benchmark_results['outcomes']
else:
    benchmark_outcomes = benchmark_results  # flat format from older runs

print(f'\nOutcomes evaluated: {list(benchmark_outcomes.keys())}')

In [ ]:
# Baseline results table: AUROC by model x outcome
MODEL_NAMES = ['xgboost', 'lr_full', 'lr_traditional', 'gbt_raw', 'ft_transformer']
DISPLAY_NAMES = {'xgboost': 'XGBoost', 'lr_full': 'LR (full)', 'lr_traditional': 'LR (trad)',
                 'gbt_raw': 'GBT (raw)', 'ft_transformer': 'FT-Trans'}

rows = []
for outcome, models in benchmark_outcomes.items():
    row = {'Outcome': outcome}
    for model in MODEL_NAMES:
        if model in models and models[model].get('auroc') is not None:
            row[DISPLAY_NAMES[model]] = models[model]['auroc']
        else:
            row[DISPLAY_NAMES[model]] = None
    rows.append(row)

baseline_df = pd.DataFrame(rows).set_index('Outcome')

# Highlight best per row
def highlight_max(s):
    is_max = s == s.max()
    return ['font-weight: bold; background-color: #d4edda' if v else '' for v in is_max]

baseline_df.style.apply(highlight_max, axis=1).format('{:.4f}', na_rep='--')

In [ ]:
# Grouped bar chart: baseline AUROC by outcome
plot_df = baseline_df.dropna(how='all')
if not plot_df.empty:
    ax = plot_df.plot(kind='bar', figsize=(12, 5), width=0.8, edgecolor='white')
    ax.set_ylabel('Test AUROC')
    ax.set_title('Baseline Benchmark: Test AUROC by Outcome')
    ax.set_ylim(0.5, max(plot_df.max().max() + 0.05, 0.8))
    ax.legend(loc='upper right', fontsize=9)
    ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5, label='random')
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    plt.show()
else:
    print('No baseline results to plot.')

## 3. FT-Transformer Hyperparameter Sweep

Grid search over:
- **Learning rate:** 1e-3, 3e-4, 1e-4, 3e-5
- **n_layers:** 2, 3, 4
- **d_token:** 128, 192, 256

36 configurations, early-stopped on val AUROC. Select best config by
average val AUROC across all evaluable outcomes.

**Exit criterion:** Tuned FT-Transformer within 0.02 AUROC of best baseline on each outcome.

In [ ]:
%%time
from itertools import product
from fin_jepa.training.train_study0 import _cfg
from fin_jepa.utils.reproducibility import seed_everything

# Sweep grid
SWEEP_LRS = [1e-3, 3e-4, 1e-4, 3e-5]
SWEEP_LAYERS = [2, 3, 4]
SWEEP_DTOKENS = [128, 192, 256]

print(f'Grid: {len(SWEEP_LRS)} lr x {len(SWEEP_LAYERS)} layers x {len(SWEEP_DTOKENS)} d_token '
      f'= {len(SWEEP_LRS) * len(SWEEP_LAYERS) * len(SWEEP_DTOKENS)} configs')

if not FORCE_RERUN and SWEEP_PATH.exists():
    print(f'Loading cached sweep results from {SWEEP_PATH}')
    with open(SWEEP_PATH) as f:
        sweep_results = json.load(f)
else:
    from fin_jepa.data.feature_engineering import (
        CATEGORICAL_FEATURES, N_SECTORS, FeatureConfig, build_feature_matrix,
    )
    from fin_jepa.data.labels import load_label_database
    from fin_jepa.data.splits import SplitConfig
    from fin_jepa.data.xbrl_loader import load_xbrl_features
    from fin_jepa.training.ablations import _train_and_evaluate

    seed = int(_cfg(benchmark_cfg, 'training.seed', 42))
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # Load data once
    raw_dir = Path(_cfg(benchmark_cfg, 'data.raw_dir', 'data/raw'))
    processed_dir = Path(_cfg(benchmark_cfg, 'data.processed_dir', 'data/processed'))

    xbrl_df = load_xbrl_features(raw_dir)
    labels_df, _ = load_label_database(processed_dir / 'label_database.parquet')
    xbrl_df['period_end'] = pd.to_datetime(xbrl_df['period_end'])
    labels_df['period_end'] = pd.to_datetime(labels_df['period_end'])
    merged = xbrl_df.merge(labels_df, on=['cik', 'period_end'], how='inner', suffixes=('', '_label'))

    split_cfg = SplitConfig(
        train_end=_cfg(benchmark_cfg, 'data.split.train_end', '2017-12-31'),
        val_end=_cfg(benchmark_cfg, 'data.split.val_end', '2019-12-31'),
        test_end=_cfg(benchmark_cfg, 'data.split.test_end', '2023-12-31'),
    )
    feat_cfg = FeatureConfig(
        use_raw=_cfg(benchmark_cfg, 'features.use_raw', True),
        use_ratios=_cfg(benchmark_cfg, 'features.use_ratios', True),
        use_yoy=_cfg(benchmark_cfg, 'features.use_yoy', True),
        use_sic=_cfg(benchmark_cfg, 'features.use_sic', True),
        use_missingness_flags=_cfg(benchmark_cfg, 'features.use_missingness_flags', True),
        coverage_threshold=_cfg(benchmark_cfg, 'features.coverage_threshold', 0.50),
        normalization_method=_cfg(benchmark_cfg, 'features.normalization_method', 'quantile'),
        median_impute=_cfg(benchmark_cfg, 'features.median_impute', True),
    )

    universe_df = None
    universe_path = raw_dir / 'company_universe.parquet'
    if universe_path.exists() and feat_cfg.use_sic:
        universe_df = pd.read_parquet(universe_path)

    splits, scaler, feature_cols, categorical_cols = build_feature_matrix(
        merged, split_cfg, feat_cfg, universe_df=universe_df,
    )
    n_features = len(feature_cols)
    n_cat = len(categorical_cols)
    cat_cards = [N_SECTORS] * n_cat if n_cat > 0 else None

    # Run sweep
    sweep_results = {'configs': [], 'best_config': None}
    total = len(SWEEP_LRS) * len(SWEEP_LAYERS) * len(SWEEP_DTOKENS)

    for i, (lr, n_layers, d_token) in enumerate(product(SWEEP_LRS, SWEEP_LAYERS, SWEEP_DTOKENS)):
        # n_heads must divide d_token
        n_heads = 8 if d_token % 8 == 0 else 4
        config_name = f'lr={lr:.0e}_layers={n_layers}_d={d_token}'
        print(f'\n[{i+1}/{total}] {config_name}')

        model_kwargs = {
            'n_features': n_features,
            'd_token': d_token,
            'n_heads': n_heads,
            'n_layers': n_layers,
            'd_ffn_factor': 4,
            'dropout': 0.0,
            'n_outputs': 1,
            'n_cat_features': n_cat,
            'cat_cardinalities': cat_cards,
        }

        config_result = {
            'lr': lr, 'n_layers': n_layers, 'd_token': d_token, 'n_heads': n_heads,
            'outcomes': {},
        }

        for outcome in OUTCOMES:
            seed_everything(seed)
            metrics = _train_and_evaluate(
                splits, feature_cols, outcome, device, model_kwargs,
                cat_feature_cols=categorical_cols,
            )
            config_result['outcomes'][outcome] = metrics
            auroc = metrics.get('auroc')
            print(f'  {outcome}: {auroc:.4f}' if auroc else f'  {outcome}: skipped')

        # Average val AUROC across evaluable outcomes
        aurocs = [m['auroc'] for m in config_result['outcomes'].values()
                  if m.get('auroc') is not None]
        config_result['mean_auroc'] = float(np.mean(aurocs)) if aurocs else 0.0
        sweep_results['configs'].append(config_result)

    # Select best config by mean AUROC
    best = max(sweep_results['configs'], key=lambda c: c['mean_auroc'])
    sweep_results['best_config'] = {
        'lr': best['lr'], 'n_layers': best['n_layers'],
        'd_token': best['d_token'], 'n_heads': best['n_heads'],
        'mean_auroc': best['mean_auroc'],
    }

    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    with open(SWEEP_PATH, 'w') as f:
        json.dump(sweep_results, f, indent=2, default=str)
    print(f'\nSweep results saved to {SWEEP_PATH}')

best_cfg = sweep_results['best_config']
print(f"\nBest config: lr={best_cfg['lr']}, n_layers={best_cfg['n_layers']}, "
      f"d_token={best_cfg['d_token']}, mean_auroc={best_cfg['mean_auroc']:.4f}")

In [ ]:
# Sweep results: top 10 configs by mean AUROC
sweep_rows = []
for cfg in sweep_results['configs']:
    row = {'lr': cfg['lr'], 'n_layers': cfg['n_layers'], 'd_token': cfg['d_token'],
           'mean_auroc': cfg['mean_auroc']}
    for outcome in OUTCOMES:
        row[outcome] = cfg['outcomes'].get(outcome, {}).get('auroc')
    sweep_rows.append(row)

sweep_df = pd.DataFrame(sweep_rows).sort_values('mean_auroc', ascending=False)
sweep_df.head(10).style.format({'lr': '{:.0e}', 'mean_auroc': '{:.4f}',
    **{o: '{:.4f}' for o in OUTCOMES}}, na_rep='--').background_gradient(
    subset=['mean_auroc'], cmap='Greens')

In [ ]:
# Heatmap: mean AUROC by lr x (n_layers, d_token)
pivot_rows = []
for cfg in sweep_results['configs']:
    pivot_rows.append({
        'lr': f"{cfg['lr']:.0e}",
        'arch': f"L={cfg['n_layers']} D={cfg['d_token']}",
        'mean_auroc': cfg['mean_auroc'],
    })

pivot_df = pd.DataFrame(pivot_rows).pivot(index='lr', columns='arch', values='mean_auroc')

fig, ax = plt.subplots(figsize=(12, 4))
sns.heatmap(pivot_df, annot=True, fmt='.4f', cmap='YlOrRd', ax=ax,
            linewidths=0.5, cbar_kws={'label': 'Mean Test AUROC'})
ax.set_title('FT-Transformer Sweep: Mean AUROC by LR x Architecture')
ax.set_ylabel('Learning Rate')
ax.set_xlabel('Architecture (Layers, d_token)')
plt.tight_layout()
plt.show()

## 4. SSL Pretraining with Tuned Architecture

Re-run SSL pretraining (mask ratios 0.15, 0.30, 0.50) using the best
hyperparameters from the sweep, then fine-tune on all outcomes.

In [ ]:
%%time
if not FORCE_RERUN and SSL_V2_PATH.exists():
    print(f'Loading cached SSL v2 results from {SSL_V2_PATH}')
    with open(SSL_V2_PATH) as f:
        ssl_results = json.load(f)
else:
    # Build tuned SSL config from best sweep params
    best = sweep_results['best_config']
    ssl_overrides = {
        'ft_transformer': {
            'd_token': best['d_token'],
            'n_heads': best['n_heads'],
            'n_layers': best['n_layers'],
        },
        'ssl_experiment': {'force_pretrain': True},
        'checkpoint_dir': 'models/checkpoints/study0_ssl_experiment_v2',
    }
    ssl_cfg_v2 = OmegaConf.merge(ssl_cfg, ssl_overrides)

    from fin_jepa.training.pretrain_ssl import run_ssl_experiment
    ssl_results = run_ssl_experiment(ssl_cfg_v2)

    # Save as v2
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    with open(SSL_V2_PATH, 'w') as f:
        json.dump(ssl_results, f, indent=2, default=str)
    print('SSL v2 experiment complete.')

print(f"\nSSL Recommendation: {ssl_results['recommendation']}")
print(f"Architecture: d_token={best_cfg['d_token']}, n_layers={best_cfg['n_layers']}, "
      f"n_heads={best_cfg['n_heads']}")

In [ ]:
# SSL v2 results table: outcome x (FT scratch, each mask ratio, best delta)
mask_ratios = sorted(ssl_results.get('pretrained', {}).keys())

rows = []
for outcome in OUTCOMES:
    baseline_auroc = ssl_results.get('baseline', {}).get(outcome, {}).get('auroc')
    row = {'Outcome': outcome, 'FT tuned (scratch)': baseline_auroc}

    best_auroc, best_mr = baseline_auroc, 'none'
    for mr in mask_ratios:
        auroc = ssl_results['pretrained'].get(mr, {}).get(outcome, {}).get('auroc')
        row[f'mr={mr}'] = auroc
        if auroc is not None and (best_auroc is None or auroc > best_auroc):
            best_auroc = auroc
            best_mr = mr

    delta = (best_auroc - baseline_auroc) if baseline_auroc and best_auroc else None
    row['Best MR'] = best_mr
    row['Delta'] = delta
    rows.append(row)

ssl_df = pd.DataFrame(rows).set_index('Outcome')
ssl_df.style.format('{:.4f}', na_rep='--', subset=[c for c in ssl_df.columns if c != 'Best MR'])

In [ ]:
# SSL pretraining loss curves
loss_curves = ssl_results.get('loss_curves', {})

if any(len(v) > 0 for v in loss_curves.values()):
    fig, ax = plt.subplots(figsize=(10, 4))
    for mr, losses in sorted(loss_curves.items()):
        if losses:
            ax.plot(range(1, len(losses) + 1), losses, label=f'mr={mr}')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Reconstruction Loss (MSE)')
    ax.set_title('SSL Pretraining Loss Curves (Tuned Architecture)')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print('No loss curves available (loaded from cache).')

## 5. Final Comparison: All Models

Head-to-head on the test set with all contestants:
- LR (traditional ratios)
- LR (full features)
- XGBoost (full features)
- GBT (raw XBRL)
- FT-Transformer (tuned, scratch)
- FT-Transformer (tuned, best SSL pretrained)

In [ ]:
# Build unified comparison table
ALL_MODELS = ['XGBoost', 'LR (full)', 'LR (trad)', 'GBT (raw)',
              'FT tuned', 'FT tuned+SSL']

# Get tuned scratch results from the sweep best config
best_sweep = max(sweep_results['configs'], key=lambda c: c['mean_auroc'])

rows = []
for outcome in OUTCOMES:
    row = {'Outcome': outcome}

    # From baseline benchmark
    bm = benchmark_outcomes.get(outcome, {})
    row['XGBoost'] = bm.get('xgboost', {}).get('auroc')
    row['LR (full)'] = bm.get('lr_full', {}).get('auroc')
    row['LR (trad)'] = bm.get('lr_traditional', {}).get('auroc')
    row['GBT (raw)'] = bm.get('gbt_raw', {}).get('auroc')

    # FT tuned scratch: from sweep best config
    row['FT tuned'] = best_sweep['outcomes'].get(outcome, {}).get('auroc')

    # Best SSL-pretrained FT-Transformer (tuned architecture)
    best_ssl, best_mr = None, 'none'
    for mr in mask_ratios:
        auroc = ssl_results.get('pretrained', {}).get(mr, {}).get(outcome, {}).get('auroc')
        if auroc is not None and (best_ssl is None or auroc > best_ssl):
            best_ssl = auroc
            best_mr = mr
    row['FT tuned+SSL'] = best_ssl
    row['Best MR'] = best_mr

    rows.append(row)

combined_df = pd.DataFrame(rows).set_index('Outcome')
display_cols = [c for c in combined_df.columns if c != 'Best MR']

combined_df[display_cols].style.apply(highlight_max, axis=1).format('{:.4f}', na_rep='--')

In [ ]:
# Hero chart: all models compared
plot_cols = [c for c in ALL_MODELS if c in combined_df.columns]
plot_data = combined_df[plot_cols].dropna(how='all')

if not plot_data.empty:
    colors = ['#4e79a7', '#f28e2b', '#e15759', '#76b7b2', '#59a14f', '#edc948']
    ax = plot_data.plot(kind='bar', figsize=(14, 6), width=0.85,
                        color=colors[:len(plot_cols)], edgecolor='white')
    ax.set_ylabel('Test AUROC')
    ax.set_title('Study 0 Final Benchmark: All Models (Test Set)')
    ax.set_ylim(0.5, max(plot_data.max().max() + 0.05, 0.8))
    ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5)
    ax.legend(loc='upper right', fontsize=9)
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    plt.show()
else:
    print('No combined results to plot.')

In [ ]:
%%time
# Re-generate test predictions for bootstrap CIs
# We need raw (y_true, y_score) arrays for each model x outcome

FINAL_RESULT_PATH = RESULTS_DIR / 'final_benchmark.json'

if not FORCE_RERUN and FINAL_RESULT_PATH.exists():
    print(f'Loading cached final benchmark from {FINAL_RESULT_PATH}')
    with open(FINAL_RESULT_PATH) as f:
        final_results = json.load(f)
else:
    from fin_jepa.data.feature_engineering import (
        CATEGORICAL_FEATURES, N_SECTORS, FeatureConfig, build_feature_matrix,
    )
    from fin_jepa.data.labels import load_label_database
    from fin_jepa.data.splits import SplitConfig
    from fin_jepa.data.xbrl_loader import load_xbrl_features
    from fin_jepa.models.ft_transformer import FTTransformer
    from fin_jepa.training.dataset import make_dataloader
    from fin_jepa.training.train_study0 import _predict_scores, train_ft_transformer, MIN_POSITIVES
    import torch.nn as nn
    import xgboost as xgb

    seed = int(_cfg(benchmark_cfg, 'training.seed', 42))
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # Load data if not already loaded
    if 'splits' not in dir() or splits is None:
        raw_dir = Path(_cfg(benchmark_cfg, 'data.raw_dir', 'data/raw'))
        processed_dir = Path(_cfg(benchmark_cfg, 'data.processed_dir', 'data/processed'))
        xbrl_df = load_xbrl_features(raw_dir)
        labels_df, _ = load_label_database(processed_dir / 'label_database.parquet')
        xbrl_df['period_end'] = pd.to_datetime(xbrl_df['period_end'])
        labels_df['period_end'] = pd.to_datetime(labels_df['period_end'])
        merged = xbrl_df.merge(labels_df, on=['cik', 'period_end'], how='inner', suffixes=('', '_label'))
        split_cfg = SplitConfig(
            train_end=_cfg(benchmark_cfg, 'data.split.train_end', '2017-12-31'),
            val_end=_cfg(benchmark_cfg, 'data.split.val_end', '2019-12-31'),
            test_end=_cfg(benchmark_cfg, 'data.split.test_end', '2023-12-31'),
        )
        feat_cfg = FeatureConfig(
            use_raw=_cfg(benchmark_cfg, 'features.use_raw', True),
            use_ratios=_cfg(benchmark_cfg, 'features.use_ratios', True),
            use_yoy=_cfg(benchmark_cfg, 'features.use_yoy', True),
            use_sic=_cfg(benchmark_cfg, 'features.use_sic', True),
            use_missingness_flags=_cfg(benchmark_cfg, 'features.use_missingness_flags', True),
            coverage_threshold=_cfg(benchmark_cfg, 'features.coverage_threshold', 0.50),
            normalization_method=_cfg(benchmark_cfg, 'features.normalization_method', 'quantile'),
            median_impute=_cfg(benchmark_cfg, 'features.median_impute', True),
        )
        universe_df = None
        universe_path = raw_dir / 'company_universe.parquet'
        if universe_path.exists() and feat_cfg.use_sic:
            universe_df = pd.read_parquet(universe_path)
        splits, scaler, feature_cols, categorical_cols = build_feature_matrix(
            merged, split_cfg, feat_cfg, universe_df=universe_df,
        )
        n_features = len(feature_cols)
        n_cat = len(categorical_cols)
        cat_cards = [N_SECTORS] * n_cat if n_cat > 0 else None

    best = sweep_results['best_config']
    model_kwargs = {
        'n_features': n_features,
        'd_token': best['d_token'],
        'n_heads': best['n_heads'],
        'n_layers': best['n_layers'],
        'd_ffn_factor': 4,
        'dropout': 0.0,
        'n_outputs': 1,
        'n_cat_features': n_cat,
        'cat_cardinalities': cat_cards,
    }

    # Load best SSL checkpoint
    ssl_ckpt_dir = Path('models/checkpoints/study0_ssl_experiment_v2')
    best_mr_key = None
    best_mr_auroc = -1
    for mr in mask_ratios:
        aurocs = [ssl_results['pretrained'].get(mr, {}).get(o, {}).get('auroc', 0)
                  for o in OUTCOMES]
        avg = np.mean([a for a in aurocs if a and a > 0]) if any(a and a > 0 for a in aurocs) else 0
        if avg > best_mr_auroc:
            best_mr_auroc = avg
            best_mr_key = mr

    ssl_ckpt_path = ssl_ckpt_dir / f'encoder_mr{best_mr_key}.pt'
    ssl_state_dict = torch.load(ssl_ckpt_path, map_location=device) if ssl_ckpt_path.exists() else None

    # For each outcome: train tuned FT (scratch + SSL), collect test predictions
    final_results = {'gate_scratch': {}, 'gate_ssl': {}, 'gate_xgb': {}, 'bootstrap': {}}
    batch_size = 256
    _cat_cols = categorical_cols or None

    for outcome in OUTCOMES:
        test_df = splits['test'][splits['test'][outcome].notna()]
        train_df = splits['train'][splits['train'][outcome].notna()]
        val_df = splits['val'][splits['val'][outcome].notna()]
        n_pos = int(train_df[outcome].sum())
        if n_pos < MIN_POSITIVES:
            print(f'{outcome}: skipped (< {MIN_POSITIVES} positives)')
            continue

        n_neg = len(train_df) - n_pos
        pos_weight = n_neg / max(n_pos, 1)

        train_loader = make_dataloader(train_df, feature_cols, outcome, batch_size, shuffle=True, cat_feature_cols=_cat_cols)
        val_loader = make_dataloader(val_df, feature_cols, outcome, batch_size, shuffle=False, cat_feature_cols=_cat_cols)
        test_loader = make_dataloader(test_df, feature_cols, outcome, batch_size, shuffle=False, cat_feature_cols=_cat_cols)

        # XGBoost: retrain to get raw test predictions for bootstrap
        X_train = train_df[feature_cols].values
        y_train = train_df[outcome].values
        X_test = test_df[feature_cols].values
        y_test = test_df[outcome].values

        xgb_params = {
            'n_estimators': int(_cfg(benchmark_cfg, 'xgboost.n_estimators', 500)),
            'learning_rate': float(_cfg(benchmark_cfg, 'xgboost.learning_rate', 0.05)),
            'max_depth': int(_cfg(benchmark_cfg, 'xgboost.max_depth', 6)),
            'subsample': float(_cfg(benchmark_cfg, 'xgboost.subsample', 0.8)),
            'colsample_bytree': float(_cfg(benchmark_cfg, 'xgboost.colsample_bytree', 0.8)),
            'scale_pos_weight': pos_weight,
            'eval_metric': 'auc',
            'random_state': 42,
            'tree_method': 'hist',
        }
        xgb_model = xgb.XGBClassifier(**xgb_params)
        xgb_model.fit(X_train, y_train)
        xgb_scores = xgb_model.predict_proba(X_test)[:, 1]
        xgb_auroc = roc_auc_score(y_test, xgb_scores)
        final_results['gate_xgb'][outcome] = {'auroc': float(xgb_auroc)}

        # FT-Transformer tuned (scratch)
        seed_everything(seed)
        model_scratch = FTTransformer(**model_kwargs).to(device)
        optimizer = torch.optim.AdamW(model_scratch.parameters(), lr=best['lr'], weight_decay=1e-5)
        criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_weight], device=device))
        train_ft_transformer(model_scratch, train_loader, val_loader, criterion, optimizer, device, epochs=100, patience=10)
        yt_scratch, ys_scratch = _predict_scores(model_scratch, test_loader, device)
        ft_scratch_auroc = roc_auc_score(yt_scratch, ys_scratch)
        final_results['gate_scratch'][outcome] = {'auroc': float(ft_scratch_auroc)}

        # Bootstrap CI: FT scratch vs XGBoost
        mean_d, ci_lo, ci_hi = bootstrap_auroc_delta(
            yt_scratch, ys_scratch, y_test, xgb_scores, n_bootstrap=1000)
        final_results['bootstrap'][outcome] = {
            'scratch_vs_xgb': {'mean': mean_d, 'ci_low': ci_lo, 'ci_high': ci_hi},
        }

        # FT-Transformer tuned + SSL
        if ssl_state_dict is not None:
            seed_everything(seed)
            model_ssl = FTTransformer(**model_kwargs).to(device)
            model_ssl.load_state_dict(ssl_state_dict, strict=False)
            optimizer_ssl = torch.optim.AdamW(model_ssl.parameters(), lr=best['lr'], weight_decay=1e-5)
            criterion_ssl = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_weight], device=device))
            train_ft_transformer(model_ssl, train_loader, val_loader, criterion_ssl, optimizer_ssl, device, epochs=100, patience=10)
            yt_ssl, ys_ssl = _predict_scores(model_ssl, test_loader, device)
            ft_ssl_auroc = roc_auc_score(yt_ssl, ys_ssl)
            final_results['gate_ssl'][outcome] = {'auroc': float(ft_ssl_auroc)}

            mean_d2, ci_lo2, ci_hi2 = bootstrap_auroc_delta(
                yt_ssl, ys_ssl, y_test, xgb_scores, n_bootstrap=1000)
            final_results['bootstrap'][outcome]['ssl_vs_xgb'] = {
                'mean': mean_d2, 'ci_low': ci_lo2, 'ci_high': ci_hi2,
            }

        print(f'{outcome}: FT_scratch={ft_scratch_auroc:.4f}  XGB={xgb_auroc:.4f}  '
              f'delta={ft_scratch_auroc - xgb_auroc:+.4f}  '
              f'95%CI=[{ci_lo:+.4f}, {ci_hi:+.4f}]')

    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    with open(FINAL_RESULT_PATH, 'w') as f:
        json.dump(final_results, f, indent=2, default=str)
    print(f'\nFinal benchmark saved to {FINAL_RESULT_PATH}')

## 6. Go/No-Go Gate with Bootstrap CIs

**Rule:** FT-Transformer must beat XGBoost on ≥3 of 5 outcomes by ≥0.01 AUROC on the test set.

We evaluate with:
1. FT-Transformer (tuned, scratch) vs XGBoost
2. FT-Transformer (tuned, best SSL) vs XGBoost

Bootstrap confidence intervals (1000 resamples) on AUROC differences.

In [ ]:
from fin_jepa.training.metrics import go_no_go_gate
from sklearn.metrics import roc_auc_score

def bootstrap_auroc_delta(y_true_ft, y_score_ft, y_true_xgb, y_score_xgb,
                          n_bootstrap=1000, seed=42):
    """Bootstrap CI for AUROC(FT) - AUROC(XGB).

    Assumes paired samples (same test set). Returns (mean_delta, ci_low, ci_high).
    """
    rng = np.random.RandomState(seed)
    n = len(y_true_ft)
    deltas = []
    for _ in range(n_bootstrap):
        idx = rng.choice(n, size=n, replace=True)
        yt_ft, ys_ft = y_true_ft[idx], y_score_ft[idx]
        yt_xgb, ys_xgb = y_true_xgb[idx], y_score_xgb[idx]
        # Need both classes present in resample
        if len(np.unique(yt_ft)) < 2 or len(np.unique(yt_xgb)) < 2:
            continue
        auc_ft = roc_auc_score(yt_ft, ys_ft)
        auc_xgb = roc_auc_score(yt_xgb, ys_xgb)
        deltas.append(auc_ft - auc_xgb)
    deltas = np.array(deltas)
    return float(np.mean(deltas)), float(np.percentile(deltas, 2.5)), float(np.percentile(deltas, 97.5))

print('Bootstrap function defined. Predictions needed for CI computation.')

In [ ]:
%%time
# Re-generate test predictions for bootstrap CIs
# We need raw (y_true, y_score) arrays for each model x outcome

FINAL_RESULT_PATH = RESULTS_DIR / 'final_benchmark.json'

if not FORCE_RERUN and FINAL_RESULT_PATH.exists():
    print(f'Loading cached final benchmark from {FINAL_RESULT_PATH}')
    with open(FINAL_RESULT_PATH) as f:
        final_results = json.load(f)
else:
    from fin_jepa.data.feature_engineering import (
        CATEGORICAL_FEATURES, N_SECTORS, FeatureConfig, build_feature_matrix,
    )
    from fin_jepa.data.labels import load_label_database
    from fin_jepa.data.splits import SplitConfig
    from fin_jepa.data.xbrl_loader import load_xbrl_features
    from fin_jepa.models.ft_transformer import FTTransformer
    from fin_jepa.training.dataset import make_dataloader
    from fin_jepa.training.train_study0 import _predict_scores, train_ft_transformer, MIN_POSITIVES
    import torch.nn as nn

    seed = int(_cfg(benchmark_cfg, 'training.seed', 42))
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # Load data if not already loaded
    if 'splits' not in dir() or splits is None:
        raw_dir = Path(_cfg(benchmark_cfg, 'data.raw_dir', 'data/raw'))
        processed_dir = Path(_cfg(benchmark_cfg, 'data.processed_dir', 'data/processed'))
        xbrl_df = load_xbrl_features(raw_dir)
        labels_df, _ = load_label_database(processed_dir / 'label_database.parquet')
        xbrl_df['period_end'] = pd.to_datetime(xbrl_df['period_end'])
        labels_df['period_end'] = pd.to_datetime(labels_df['period_end'])
        merged = xbrl_df.merge(labels_df, on=['cik', 'period_end'], how='inner', suffixes=('', '_label'))
        split_cfg = SplitConfig(
            train_end=_cfg(benchmark_cfg, 'data.split.train_end', '2017-12-31'),
            val_end=_cfg(benchmark_cfg, 'data.split.val_end', '2019-12-31'),
            test_end=_cfg(benchmark_cfg, 'data.split.test_end', '2023-12-31'),
        )
        feat_cfg = FeatureConfig(
            use_raw=_cfg(benchmark_cfg, 'features.use_raw', True),
            use_ratios=_cfg(benchmark_cfg, 'features.use_ratios', True),
            use_yoy=_cfg(benchmark_cfg, 'features.use_yoy', True),
            use_sic=_cfg(benchmark_cfg, 'features.use_sic', True),
            use_missingness_flags=_cfg(benchmark_cfg, 'features.use_missingness_flags', True),
            coverage_threshold=_cfg(benchmark_cfg, 'features.coverage_threshold', 0.50),
            normalization_method=_cfg(benchmark_cfg, 'features.normalization_method', 'quantile'),
            median_impute=_cfg(benchmark_cfg, 'features.median_impute', True),
        )
        universe_df = None
        universe_path = raw_dir / 'company_universe.parquet'
        if universe_path.exists() and feat_cfg.use_sic:
            universe_df = pd.read_parquet(universe_path)
        splits, scaler, feature_cols, categorical_cols = build_feature_matrix(
            merged, split_cfg, feat_cfg, universe_df=universe_df,
        )
        n_features = len(feature_cols)
        n_cat = len(categorical_cols)
        cat_cards = [N_SECTORS] * n_cat if n_cat > 0 else None

    best = sweep_results['best_config']
    model_kwargs = {
        'n_features': n_features,
        'd_token': best['d_token'],
        'n_heads': best['n_heads'],
        'n_layers': best['n_layers'],
        'd_ffn_factor': 4,
        'dropout': 0.0,
        'n_outputs': 1,
        'n_cat_features': n_cat,
        'cat_cardinalities': cat_cards,
    }

    # Load best SSL checkpoint
    ssl_ckpt_dir = Path('models/checkpoints/study0_ssl_experiment_v2')
    best_mr_key = None
    best_mr_auroc = -1
    for mr in mask_ratios:
        aurocs = [ssl_results['pretrained'].get(mr, {}).get(o, {}).get('auroc', 0)
                  for o in OUTCOMES]
        avg = np.mean([a for a in aurocs if a and a > 0]) if any(a and a > 0 for a in aurocs) else 0
        if avg > best_mr_auroc:
            best_mr_auroc = avg
            best_mr_key = mr

    ssl_ckpt_path = ssl_ckpt_dir / f'encoder_mr{best_mr_key}.pt'
    ssl_state_dict = torch.load(ssl_ckpt_path, map_location=device) if ssl_ckpt_path.exists() else None

    # For each outcome: train tuned FT (scratch + SSL), collect test predictions
    final_results = {'gate_scratch': {}, 'gate_ssl': {}, 'bootstrap': {}}
    batch_size = 256
    _cat_cols = categorical_cols or None

    for outcome in OUTCOMES:
        test_df = splits['test'][splits['test'][outcome].notna()]
        train_df = splits['train'][splits['train'][outcome].notna()]
        val_df = splits['val'][splits['val'][outcome].notna()]
        n_pos = int(train_df[outcome].sum())
        if n_pos < MIN_POSITIVES:
            print(f'{outcome}: skipped (< {MIN_POSITIVES} positives)')
            continue

        n_neg = len(train_df) - n_pos
        pos_weight = n_neg / max(n_pos, 1)

        train_loader = make_dataloader(train_df, feature_cols, outcome, batch_size, shuffle=True, cat_feature_cols=_cat_cols)
        val_loader = make_dataloader(val_df, feature_cols, outcome, batch_size, shuffle=False, cat_feature_cols=_cat_cols)
        test_loader = make_dataloader(test_df, feature_cols, outcome, batch_size, shuffle=False, cat_feature_cols=_cat_cols)

        # XGBoost predictions (from baseline benchmark — we store the AUROC, not raw preds)
        # For bootstrap we need to retrain XGBoost to get raw predictions
        from fin_jepa.training.train_study0 import _build_xgboost
        import xgboost as xgb
        X_train = train_df[feature_cols].values
        y_train = train_df[outcome].values
        X_test = test_df[feature_cols].values
        y_test = test_df[outcome].values

        xgb_params = {
            'n_estimators': int(_cfg(benchmark_cfg, 'xgboost.n_estimators', 500)),
            'learning_rate': float(_cfg(benchmark_cfg, 'xgboost.learning_rate', 0.05)),
            'max_depth': int(_cfg(benchmark_cfg, 'xgboost.max_depth', 6)),
            'subsample': float(_cfg(benchmark_cfg, 'xgboost.subsample', 0.8)),
            'colsample_bytree': float(_cfg(benchmark_cfg, 'xgboost.colsample_bytree', 0.8)),
            'scale_pos_weight': pos_weight,
            'eval_metric': 'auc',
            'random_state': 42,
            'tree_method': 'hist',
        }
        xgb_model = xgb.XGBClassifier(**xgb_params)
        xgb_model.fit(X_train, y_train)
        xgb_scores = xgb_model.predict_proba(X_test)[:, 1]

        # FT-Transformer tuned (scratch)
        seed_everything(seed)
        model_scratch = FTTransformer(**model_kwargs).to(device)
        optimizer = torch.optim.AdamW(model_scratch.parameters(), lr=best['lr'], weight_decay=1e-5)
        criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_weight], device=device))
        train_ft_transformer(model_scratch, train_loader, val_loader, criterion, optimizer, device, epochs=100, patience=10)
        yt_scratch, ys_scratch = _predict_scores(model_scratch, test_loader, device)

        # FT-Transformer tuned + SSL
        ft_ssl_scores = None
        if ssl_state_dict is not None:
            seed_everything(seed)
            model_ssl = FTTransformer(**model_kwargs).to(device)
            model_ssl.load_state_dict(ssl_state_dict, strict=False)
            optimizer_ssl = torch.optim.AdamW(model_ssl.parameters(), lr=best['lr'], weight_decay=1e-5)
            criterion_ssl = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_weight], device=device))
            train_ft_transformer(model_ssl, train_loader, val_loader, criterion_ssl, optimizer_ssl, device, epochs=100, patience=10)
            yt_ssl, ys_ssl = _predict_scores(model_ssl, test_loader, device)
            ft_ssl_scores = ys_ssl

        # Store AUROC for gate
        ft_scratch_auroc = roc_auc_score(yt_scratch, ys_scratch)
        xgb_auroc = roc_auc_score(y_test, xgb_scores)
        final_results['gate_scratch'][outcome] = {'auroc': float(ft_scratch_auroc)}
        final_results['gate_ssl'][outcome] = {}

        # Bootstrap CIs: FT scratch vs XGBoost
        mean_d, ci_lo, ci_hi = bootstrap_auroc_delta(
            yt_scratch, ys_scratch, y_test, xgb_scores, n_bootstrap=1000)
        final_results['bootstrap'][outcome] = {
            'scratch_vs_xgb': {'mean': mean_d, 'ci_low': ci_lo, 'ci_high': ci_hi},
        }

        if ft_ssl_scores is not None:
            ft_ssl_auroc = roc_auc_score(yt_ssl, ys_ssl)
            final_results['gate_ssl'][outcome] = {'auroc': float(ft_ssl_auroc)}
            mean_d2, ci_lo2, ci_hi2 = bootstrap_auroc_delta(
                yt_ssl, ys_ssl, y_test, xgb_scores, n_bootstrap=1000)
            final_results['bootstrap'][outcome]['ssl_vs_xgb'] = {
                'mean': mean_d2, 'ci_low': ci_lo2, 'ci_high': ci_hi2,
            }

        print(f'{outcome}: FT_scratch={ft_scratch_auroc:.4f}  XGB={xgb_auroc:.4f}  '
              f'delta={ft_scratch_auroc - xgb_auroc:+.4f}  '
              f'95%CI=[{ci_lo:+.4f}, {ci_hi:+.4f}]')

    # Also store XGBoost AUROCs in final results for the gate
    final_results['gate_xgb'] = {}
    for outcome in OUTCOMES:
        bm = benchmark_outcomes.get(outcome, {})
        xgb_a = bm.get('xgboost', {}).get('auroc')
        if xgb_a is not None:
            final_results['gate_xgb'][outcome] = {'auroc': xgb_a}

    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    with open(FINAL_RESULT_PATH, 'w') as f:
        json.dump(final_results, f, indent=2, default=str)
    print(f'\nFinal benchmark saved to {FINAL_RESULT_PATH}')

In [ ]:
# Go/No-Go Gate
print('=' * 65)
print('GO/NO-GO GATE: FT-Transformer (tuned) vs XGBoost')
print('Rule: FT must beat XGB on >= 3/5 outcomes by >= 0.01 AUROC')
print('=' * 65)

# Use final_results for gate (re-trained with tuned hyperparams)
xgb_gate = final_results.get('gate_xgb', {})
ft_scratch_gate = final_results.get('gate_scratch', {})
ft_ssl_gate = final_results.get('gate_ssl', {})
bootstrap = final_results.get('bootstrap', {})

gate_outcomes_s = [o for o in OUTCOMES if o in xgb_gate and o in ft_scratch_gate
                   and ft_scratch_gate[o].get('auroc') is not None]
gate_outcomes_ssl = [o for o in OUTCOMES if o in xgb_gate and o in ft_ssl_gate
                     and ft_ssl_gate[o].get('auroc') is not None]

print('\n--- Gate 1: FT-Transformer (tuned, scratch) vs XGBoost ---')
if gate_outcomes_s:
    passed_s, wins_s, detail_s = go_no_go_gate(ft_scratch_gate, xgb_gate, gate_outcomes_s)
    status_s = 'PASSED' if passed_s else 'FAILED'
    print(f'Result: {status_s} ({wins_s}/{len(gate_outcomes_s)} wins)\n')
    for o, d in detail_s.items():
        marker = '+' if d['win'] else '-'
        delta = d['ft_auroc'] - d['xgb_auroc']
        bs = bootstrap.get(o, {}).get('scratch_vs_xgb', {})
        ci_str = f"  95%CI=[{bs['ci_low']:+.4f}, {bs['ci_high']:+.4f}]" if bs else ''
        print(f'  [{marker}] {o:25s}  FT={d["ft_auroc"]:.4f}  XGB={d["xgb_auroc"]:.4f}  '
              f'delta={delta:+.4f}{ci_str}')
else:
    print('No overlapping outcomes to evaluate.')

print('\n--- Gate 2: FT-Transformer (tuned, best SSL) vs XGBoost ---')
if gate_outcomes_ssl:
    passed_ssl, wins_ssl, detail_ssl = go_no_go_gate(ft_ssl_gate, xgb_gate, gate_outcomes_ssl)
    status_ssl = 'PASSED' if passed_ssl else 'FAILED'
    print(f'Result: {status_ssl} ({wins_ssl}/{len(gate_outcomes_ssl)} wins)\n')
    for o, d in detail_ssl.items():
        marker = '+' if d['win'] else '-'
        delta = d['ft_auroc'] - d['xgb_auroc']
        bs = bootstrap.get(o, {}).get('ssl_vs_xgb', {})
        ci_str = f"  95%CI=[{bs['ci_low']:+.4f}, {bs['ci_high']:+.4f}]" if bs else ''
        print(f'  [{marker}] {o:25s}  FT={d["ft_auroc"]:.4f}  XGB={d["xgb_auroc"]:.4f}  '
              f'delta={delta:+.4f}{ci_str}')
else:
    print('No overlapping outcomes to evaluate.')

In [ ]:
# Bootstrap CI forest plot
bs_data = final_results.get('bootstrap', {})
plot_outcomes = [o for o in OUTCOMES if o in bs_data and 'scratch_vs_xgb' in bs_data[o]]

if plot_outcomes:
    fig, ax = plt.subplots(figsize=(10, len(plot_outcomes) * 0.8 + 2))
    y_pos = np.arange(len(plot_outcomes))
    width = 0.35

    for i, outcome in enumerate(plot_outcomes):
        # Scratch vs XGB
        bs_s = bs_data[outcome]['scratch_vs_xgb']
        ax.errorbar(bs_s['mean'], i - width/2,
                    xerr=[[bs_s['mean'] - bs_s['ci_low']], [bs_s['ci_high'] - bs_s['mean']]],
                    fmt='o', color='#59a14f', capsize=5, markersize=8,
                    label='FT tuned (scratch)' if i == 0 else '')

        # SSL vs XGB
        if 'ssl_vs_xgb' in bs_data[outcome]:
            bs_ssl = bs_data[outcome]['ssl_vs_xgb']
            ax.errorbar(bs_ssl['mean'], i + width/2,
                        xerr=[[bs_ssl['mean'] - bs_ssl['ci_low']], [bs_ssl['ci_high'] - bs_ssl['mean']]],
                        fmt='s', color='#edc948', capsize=5, markersize=8,
                        label='FT tuned+SSL' if i == 0 else '')

    ax.axvline(x=0, color='gray', linestyle='--', alpha=0.7)
    ax.axvline(x=0.01, color='green', linestyle=':', alpha=0.5, label='Gate margin (0.01)')
    ax.set_yticks(y_pos)
    ax.set_yticklabels(plot_outcomes)
    ax.set_xlabel('AUROC Delta (FT - XGBoost)')
    ax.set_title('Bootstrap 95% CI: FT-Transformer vs XGBoost AUROC Difference')
    ax.legend(loc='lower right')
    ax.grid(True, axis='x', alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print('No bootstrap data to plot.')

## 7. Save Results to Drive

In [ ]:
import shutil

DRIVE_RESULTS = '/content/drive/MyDrive/Fin-JEPA/results/study0'
os.makedirs(DRIVE_RESULTS, exist_ok=True)

# Copy all result JSONs
for fname in ['benchmark_results.json', 'ft_sweep_results.json',
              'ssl_experiment_results_v2.json', 'final_benchmark.json']:
    src = RESULTS_DIR / fname
    if src.exists():
        shutil.copy2(str(src), DRIVE_RESULTS)
        print(f'Copied {fname} to Drive')

# Copy SSL v2 checkpoints
for ckpt_name in ['study0_ssl_experiment_v2', 'study0_ssl_experiment']:
    ckpt_src = Path(f'models/checkpoints/{ckpt_name}')
    if ckpt_src.exists():
        drive_ckpt = f'/content/drive/MyDrive/Fin-JEPA/models/checkpoints/{ckpt_name}'
        shutil.copytree(str(ckpt_src), drive_ckpt, dirs_exist_ok=True)
        print(f'Copied {ckpt_name} checkpoints to Drive')

print('\nAll results saved to Google Drive!')